# 07 - VGGT-X MCMC-3DGS

This notebook runs the official baseline from beginning to end:

1. copy the same eight selected images used by the other experiments;
2. run VGGT-X with global alignment and export COLMAP geometry;
3. inspect the registered cameras and sparse points;
4. train CityGaussian with its MCMC-3DGS pose-optimization configuration;
5. evaluate the eight real views;
6. save a closed-orbit video and individual frames for notebook 08.

VGGT-X was designed for dense image collections. Eight images are used here for a fair sparse-view comparison, so a failure is also a meaningful experimental result.

## Important environment design

The active Colab kernel is not modified. `uv` creates two isolated environments because the official projects require different Python and Torch versions. Installation can take several minutes and compile CUDA extensions. Start from a fresh GPU runtime.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
CODE_ROOT = Path("/content/Project_Thesis_code")
REPOSITORY = "https://github.com/katlit/Project_Thesis.git"
BRANCH = "codex/hq200-example-notebook"
if CODE_ROOT.exists() and not (CODE_ROOT / ".git").is_dir(): shutil.rmtree(CODE_ROOT)
command = (["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(CODE_ROOT)]
           if not CODE_ROOT.exists() else ["git", "-C", str(CODE_ROOT), "pull", "--ff-only", "origin", BRANCH])
subprocess.run(command, check=True)
sys.path.insert(0, str(CODE_ROOT / "code"))
PROJECT_ROOT = Path("/content/drive/MyDrive/ITU/3D/Thesis")

import json, os, shutil, subprocess, textwrap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import Video, display

if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise RuntimeError("Connect a GPU runtime before running notebook 07.")

DATASET = "3DRealCar"
SCENE = None
FRAMES_BETWEEN = 10
MAX_GAUSSIANS = 300_000
DOWN_SAMPLE_FACTOR = 1
RUN_INSTALL = True
RUN_VGGT_X = True
RUN_TRAINING = True
RUN_EVALUATION = True
RUN_ORBIT_RENDER = True

manifest = pd.read_csv(PROJECT_ROOT / "data_processed/method_inputs/manifest.csv")
rows = manifest.query("method == 'vggt' and split == 'train' and dataset == @DATASET").copy()
SCENE = SCENE or sorted(rows.scene.unique())[0]
rows = rows[rows.scene.eq(SCENE)].sort_values(["view_order", "source"])
assert len(rows) == 8, f"Expected eight selected images, found {len(rows)}"

STAGE = Path("/content/vggtx_data") / SCENE
VGGT_X_OUTPUT = STAGE.parent / f"{SCENE}_vggt_x"
FINAL_ROOT = PROJECT_ROOT / "experiments/3DGS/VGGT_X_MCMC" / DATASET / SCENE
FINAL_ROOT.mkdir(parents=True, exist_ok=True)
print("Scene:", SCENE)
print("Final Drive folder:", FINAL_ROOT)

## 1. Install the official projects

The two projects use isolated Python environments. CityGaussian's original PyTorch 2.0.1/CUDA 11.8 pin cannot compile CUDA extensions against current Colab CUDA 12.8. The notebook therefore installs official PyTorch 2.7.1/CUDA 12.8 and transparently replaces only the obsolete `simple-knn` **one-time initial Gaussian-scale calculation** with SciPy nearest-neighbour distances. The official VGGT-X alignment, CityGaussian MCMC optimization, and gsplat renderer are unchanged. Both adaptations are saved to the experiment folder.

In [ ]:
%pip -q install uv

VGGT_X_ROOT = Path("/content/VGGT-X")
CITY_ROOT = Path("/content/CityGaussian")
VGGT_ENV = Path("/content/envs/vggt_x")
CITY_ENV = Path("/content/envs/citygaussian")

def run(command, cwd=None, env=None):
    print("RUN:", " ".join(map(str, command)))
    result = subprocess.run(
        [str(item) for item in command], cwd=cwd, env=env,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if result.returncode != 0:
        print("\n----- command output (last 12,000 characters) -----")
        print(result.stdout[-12_000:])
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}. "
            "The useful package/build error is printed immediately above."
        )
    if result.stdout.strip():
        print(result.stdout[-2_000:])

if RUN_INSTALL:
    from src.citygaussian_bridge import patch_portable_knn_initialization

    if not (VGGT_X_ROOT / ".git").is_dir():
        run(["git", "clone", "--recursive", "https://github.com/Linketic/VGGT-X.git", VGGT_X_ROOT])
    else:
        run(["git", "-C", VGGT_X_ROOT, "pull", "--ff-only"])
        run(["git", "-C", VGGT_X_ROOT, "submodule", "update", "--init", "--recursive"])

    if not (CITY_ROOT / ".git").is_dir():
        run(["git", "clone", "--recursive", "https://github.com/Linketic/CityGaussian.git", CITY_ROOT])
    else:
        run(["git", "-C", CITY_ROOT, "pull", "--ff-only"])
        run(["git", "-C", CITY_ROOT, "submodule", "update", "--init", "--recursive"])

    vggt_python = VGGT_ENV / "bin/python"
    if not vggt_python.is_file():
        run(["uv", "venv", "--clear", "--python", "3.10", "--seed", VGGT_ENV])
    else:
        print("Reusing existing VGGT-X environment:", VGGT_ENV)
    run(["uv", "pip", "install", "--python", vggt_python, "-r", VGGT_X_ROOT / "requirements.txt"])

    city_python = CITY_ENV / "bin/python"
    city_ready = CITY_ENV / ".citygaussian_ready"
    if not city_ready.is_file():
        # Clear the incomplete environment left by an earlier failed install.
        run(["uv", "venv", "--clear", "--python", "3.9", "--seed", CITY_ENV])
        # These setup.py projects need the older packaging path and NumPy 1.x.
        run([
            city_python, "-m", "pip", "install",
            "pip<25", "setuptools<70", "wheel<0.44", "numpy<2", "ninja",
        ], cwd=CITY_ROOT)
        # Match the PyTorch CUDA build to Colab's CUDA 12.8 compiler. The
        # repository's original torch 2.0.1+cu118 pin cannot build gsplat here.
        run([
            city_python, "-m", "pip", "install",
            "torch==2.7.1", "torchvision==0.22.1",
            "--index-url", "https://download.pytorch.org/whl/cu128",
        ], cwd=CITY_ROOT)
        # The selected MCMC configuration uses GSplatCameraOptRenderer, not the
        # legacy diff-gaussian rasterizer. Install the ordinary dependencies
        # without that unused CUDA package or the optional Open3D viewer.
        common_lines = (CITY_ROOT / "requirements/common.txt").read_text(encoding="utf-8").splitlines()
        skip_common = ("open3d", "git+https://github.com/graphdeco-inria/diff-gaussian-rasterization", "git+https://github.com/yzslab/simple-knn")
        filtered_common = [line for line in common_lines if not line.strip().lower().startswith(skip_common)]
        common_file = Path("/content/citygaussian_common_colab.txt")
        common_file.write_text("\n".join(filtered_common) + "\n", encoding="utf-8")
        run([
            city_python, "-m", "pip", "install",
            "lightning[pytorch-extra]==2.3.*", "pytorch-lightning==2.3.*", "bitsandbytes==0.45.*",
            "-r", common_file,
        ], cwd=CITY_ROOT)
        adaptation = patch_portable_knn_initialization(CITY_ROOT)
        adaptation["pytorch_environment"] = {
            "torch": "2.7.1+cu128",
            "torchvision": "0.22.1+cu128",
            "reason": "match current Colab CUDA 12.8 compiler for gsplat",
        }
        (FINAL_ROOT / "environment_adaptations.json").write_text(
            json.dumps(adaptation, indent=2), encoding="utf-8"
        )
        print("CityGaussian compatibility adaptation:", adaptation["status"])
        run([city_python, "-m", "pip", "install", "--no-build-isolation", "-r", "requirements/gsplat.txt"], cwd=CITY_ROOT)
        city_ready.write_text("ok\n", encoding="utf-8")
    else:
        print("Reusing complete CityGaussian environment:", CITY_ENV)

print("VGGT-X Python:", VGGT_ENV / "bin/python")
print("CityGaussian Python:", CITY_ENV / "bin/python")

## 2. Stage and verify the eight inputs

Only the annotated training selections are copied. Numeric filenames preserve canonical rotational order.

In [ ]:
if STAGE.exists():
    shutil.rmtree(STAGE)
(STAGE / "images").mkdir(parents=True)
fig, axes = plt.subplots(1, 8, figsize=(24, 3))
for index, row in enumerate(rows.itertuples(index=False)):
    image = Image.open(row.method_image).convert("RGB")
    image.save(STAGE / "images" / f"{index:02d}.png")
    axes[index].imshow(image); axes[index].set_title(f"view {int(row.view_order)}"); axes[index].axis("off")
plt.tight_layout(); plt.show()
print("Staged:", STAGE)

## 3. Run VGGT-X global alignment

This creates the `_vggt_x` folder containing images, COLMAP cameras, points, and `matches.pt`. The geometry must register all eight images before 3DGS training starts.

In [ ]:
if RUN_VGGT_X:
    run([
        VGGT_ENV / "bin/python", VGGT_X_ROOT / "demo_colmap.py",
        "--scene_dir", STAGE,
        "--shared_camera", "--use_ga", "--save_depth",
        "--total_frame_num", "8",
    ], cwd=VGGT_X_ROOT)

sparse_candidates = [VGGT_X_OUTPUT / "sparse/0", VGGT_X_OUTPUT / "sparse"]
SPARSE = next((path for path in sparse_candidates if (path / "cameras.bin").is_file()), None)
if SPARSE is None:
    raise FileNotFoundError(f"VGGT-X did not create a COLMAP model under {VGGT_X_OUTPUT}")
print("COLMAP model:", SPARSE)

## 4. Inspect geometry before training

This cell reads the official COLMAP result inside the VGGT-X environment and saves a small diagnostic file. The notebook then plots camera centers and a sampled point cloud.

In [ ]:
INSPECT_SCRIPT = Path("/content/inspect_vggtx.py")
INSPECT_SCRIPT.write_text(textwrap.dedent(f"""
import numpy as np, pycolmap
r = pycolmap.Reconstruction(r'{SPARSE}')
images = sorted(r.images.values(), key=lambda x: x.name)
centers = []
for image in images:
    value = image.cam_from_world
    pose = value() if callable(value) else value
    centers.append(np.asarray(pose.inverse().translation))
points = np.asarray([point.xyz for point in r.points3D.values()])
colors = np.asarray([point.color for point in r.points3D.values()]) / 255.0
np.savez(r'/content/vggtx_inspection.npz', centers=centers, points=points, colors=colors,
         registered=len(images), cameras=len(r.cameras))
"""), encoding="utf-8")
run([VGGT_ENV / "bin/python", INSPECT_SCRIPT])
inspection = np.load("/content/vggtx_inspection.npz")
print("Registered images:", int(inspection["registered"]), "/ 8")
if int(inspection["registered"]) != 8:
    raise RuntimeError("Stop: VGGT-X did not register every selected input.")

centers, points, point_colors = inspection["centers"], inspection["points"], inspection["colors"]
rng = np.random.default_rng(42)
if len(points) > 50_000:
    chosen = rng.choice(len(points), 50_000, replace=False)
    points, point_colors = points[chosen], point_colors[chosen]
fig = plt.figure(figsize=(14, 6))
ax1 = fig.add_subplot(121, projection="3d"); ax2 = fig.add_subplot(122, projection="3d")
closed = np.vstack([centers, centers[0]])
ax1.plot(*closed.T, "o-"); ax1.set_title("VGGT-X closed camera orbit")
ax2.scatter(*points.T, c=point_colors, s=.2); ax2.set_title(f"VGGT-X COLMAP points: {len(points):,} shown")
for axis in [ax1, ax2]: axis.set_box_aspect(np.ptp((closed if axis is ax1 else points), axis=0).clip(min=1e-6))
plt.tight_layout(); plt.show()

## 5. Train official CityGaussian MCMC-3DGS

The official pose-optimization configuration jointly refines the imperfect VGGT-X cameras and the Gaussians. `MAX_GAUSSIANS` controls memory use.

In [ ]:
RUN_NAME = f"{SCENE}_vggtx_mcmc"
CITY_OUTPUT = CITY_ROOT / "outputs" / RUN_NAME
if RUN_TRAINING:
    run([
        CITY_ENV / "bin/python", CITY_ROOT / "main.py", "fit",
        "--config", CITY_ROOT / "configs/colmap_pose_opt_mcmc.yaml",
        "--data.path", VGGT_X_OUTPUT,
        "--data.parser.init_args.down_sample_factor", str(DOWN_SAMPLE_FACTOR),
        "--data.parser.init_args.down_sample_rounding_mode", "round",
        "--model.density.init_args.cap_max", str(MAX_GAUSSIANS),
        "-n", RUN_NAME,
    ], cwd=CITY_ROOT)
if not (CITY_OUTPUT / "config.yaml").is_file():
    raise FileNotFoundError(f"CityGaussian training output missing: {CITY_OUTPUT}")
print("Training output:", CITY_OUTPUT)

## 6. Evaluate the real training views

These metrics measure input-view fit. They are useful diagnostics but are not held-out NVS scores.

In [ ]:
if RUN_EVALUATION:
    run([
        CITY_ENV / "bin/python", CITY_ROOT / "main.py", "test",
        "--config", CITY_OUTPUT / "config.yaml", "--save_val", "--val_train",
    ], cwd=CITY_ROOT)

# Keep the official files together on Drive. This includes the checkpoint and
# any metric/render files produced by the official test command.
DRIVE_MODEL = FINAL_ROOT / "citygaussian_output"
shutil.copytree(CITY_OUTPUT, DRIVE_MODEL, dirs_exist_ok=True)
metric_candidates = list(CITY_OUTPUT.rglob("*.csv")) + list(CITY_OUTPUT.rglob("*.json"))
print("Metric/result files found:")
for path in metric_candidates: print(" -", path.relative_to(CITY_OUTPUT))
csv_candidates = [path for path in metric_candidates if path.suffix.lower() == ".csv" and
                  any(word in path.name.lower() for word in ["metric", "result", "score"])]
if csv_candidates:
    shutil.copy2(csv_candidates[0], FINAL_ROOT / "metrics.csv")
    print("Standard metric table:", FINAL_ROOT / "metrics.csv")
else:
    print("The official test command did not expose a CSV. Its original logs remain in citygaussian_output.")

## 7. Render the same closed orbit used for comparison

The path follows the eight VGGT-X COLMAP cameras and inserts ten frames between neighboring views. CityGaussian renders both an MP4 and individual PNG files.

In [ ]:
PATH_SCRIPT = Path("/content/make_city_path.py")
PATH_SCRIPT.write_text(textwrap.dedent(f"""
import sys, pycolmap
sys.path.insert(0, r'{CODE_ROOT / "code"}')
from src.citygaussian_bridge import closed_colmap_camera_path
r = pycolmap.Reconstruction(r'{SPARSE}')
print(closed_colmap_camera_path(r, r'{FINAL_ROOT / "camera_path.json"}', frames_between={FRAMES_BETWEEN}))
"""), encoding="utf-8")
run([VGGT_ENV / "bin/python", PATH_SCRIPT])

ORBIT_VIDEO = FINAL_ROOT / "closed_orbit.mp4"
if RUN_ORBIT_RENDER:
    run([
        CITY_ENV / "bin/python", CITY_ROOT / "render.py", CITY_OUTPUT,
        "--camera-path-filename", FINAL_ROOT / "camera_path.json",
        "--output-path", ORBIT_VIDEO, "--save-images", "--disable-transform",
    ], cwd=CITY_ROOT)

generated_frames = Path(str(ORBIT_VIDEO) + "_frames")
ORBIT_FRAMES = FINAL_ROOT / "orbit_frames"
ORBIT_FRAMES.mkdir(parents=True, exist_ok=True)
for index, source in enumerate(sorted(generated_frames.glob("*.png"))):
    shutil.copy2(source, ORBIT_FRAMES / f"view_{index:03d}.png")
print("Saved video:", ORBIT_VIDEO)
print("Saved frames:", len(list(ORBIT_FRAMES.glob("view_*.png"))))
display(Video(str(ORBIT_VIDEO), embed=True, html_attributes="controls autoplay loop muted"))

## 8. Final output check

Notebook 08 needs `closed_orbit.mp4`, `orbit_frames/`, and the official CityGaussian output. Missing files are reported before you disconnect the runtime.

In [ ]:
checks = {
    "COLMAP cameras": SPARSE / "cameras.bin",
    "COLMAP images": SPARSE / "images.bin",
    "COLMAP points": SPARSE / "points3D.bin",
    "CityGaussian config": DRIVE_MODEL / "config.yaml",
    "closed orbit": ORBIT_VIDEO,
    "first comparison frame": ORBIT_FRAMES / "view_000.png",
}
for label, path in checks.items():
    print("OK     " if path.exists() else "MISSING", label, path)
if not all(path.exists() for path in checks.values()):
    raise RuntimeError("Notebook 07 is incomplete. Read the first missing item above.")